In [46]:
!pip install tree-sitter tree-sitter-python gitpython igraph tqdm rank_bm25 sentence-transformers -q

In [ ]:
# ── SWE-bench instance selector ───────────────────────────────────────────────
# Chọn instance cần build graph. Thay đổi INSTANCE_IDX rồi Run All.
#   0  →  pallets__flask-4045  (blueprint name chứa dấu chấm)
#   1  →  pallets__flask-4992  (Config.from_file binary mode)
#   2  →  pallets__flask-5063  (flask routes hiện subdomain)
#   None → HEAD hiện tại (không checkout)
INSTANCE_IDX = 1

_SWE = [
    ("pallets__flask-4045", "d8c37f43724cd9fb0870f77877b7c4c7e38a19e0"),
    ("pallets__flask-4992", "4c288bc97ea371817199908d0d9b12de9dae327e"),
    ("pallets__flask-5063", "182ce3dd15dfa3537391c3efaf9c3ff407d134d4"),
]
TARGET_COMMIT = _SWE[INSTANCE_IDX][1] if INSTANCE_IDX is not None else None
TARGET_ID     = _SWE[INSTANCE_IDX][0] if INSTANCE_IDX is not None else "HEAD"
print(f"Instance : {TARGET_ID}")
print(f"Commit   : {TARGET_COMMIT[:8] if TARGET_COMMIT else 'HEAD'}")


## 1. Clone & Initialize Repository
We start by cloning the target repository (Flask) and identifying the specific commit hash for version tracking within our graph.

In [ ]:
import os
from pathlib import Path
from git import Repo

REPO_URL  = "https://github.com/pallets/flask.git"
REPO_PATH = "/content/flask"

if Path(REPO_PATH).exists():
    print(f"Repo đã tồn tại tại {REPO_PATH}, bỏ qua clone.")
else:
    print(f"Cloning {REPO_URL} ...")
    Repo.clone_from(REPO_URL, REPO_PATH)
    print("Done.")

repo = Repo(REPO_PATH)

# Checkout đúng commit cho SWE-bench instance
if TARGET_COMMIT:
    repo.git.checkout(TARGET_COMMIT)
    print(f"Checked out : {TARGET_COMMIT[:8]}  ({TARGET_ID})")

BASE_COMMIT = repo.head.commit.hexsha[:8]
OUTPUT_FILE = f"flask_graph_{BASE_COMMIT}.json"
print(f"Commit     : {BASE_COMMIT}")
print(f"Output     : {OUTPUT_FILE}")


## 2. Source File Discovery
We list all Python source files using `git ls-files`, which ensures we respect `.gitignore` rules and only index relevant code.

In [48]:
import subprocess

def get_python_files(repo_path: str) -> list[str]:
    """Dùng git ls-files để liệt kê .py files (tự động theo .gitignore)."""
    try:
        out = subprocess.check_output(
            ["git", "ls-files", "--cached", "--others", "--exclude-standard"],
            cwd=repo_path, text=True, stderr=subprocess.DEVNULL,
        )
        return [l for l in out.splitlines() if l.endswith(".py")]
    except Exception:
        return [
            str(p.relative_to(repo_path)).replace("\\", "/")
            for p in Path(repo_path).rglob("*.py")
            if ".git" not in p.parts
        ]

source_files = get_python_files(REPO_PATH)
print(f"Found {len(source_files)} Python files")
print("\n".join(source_files[:10]), "...", sep="\n")

Found 83 Python files
docs/conf.py
examples/celery/make_celery.py
examples/celery/src/task_app/__init__.py
examples/celery/src/task_app/tasks.py
examples/celery/src/task_app/views.py
examples/javascript/js_example/__init__.py
examples/javascript/js_example/views.py
examples/javascript/tests/conftest.py
examples/javascript/tests/test_js_example.py
examples/tutorial/flaskr/__init__.py
...


## 3. Graph Schema & Context
This section defines our data model (Modules, Classes, Functions) and the `ParseContext` used to track state while traversing Abstract Syntax Trees (AST).

In [49]:
from __future__ import annotations
import os
from abc import ABC, abstractmethod
from dataclasses import dataclass, field, replace as dc_replace
from typing import Callable


@dataclass
class ParseContext:
    module_id:    str
    parent_id:    str
    commit:       str
    file_:        str
    directory:    str
    is_module_scope: bool
    current_class_id: str | None
    nodes_out:    list = field(default_factory=list)
    edges_out:    list = field(default_factory=list)
    call_refs_out:     list = field(default_factory=list)
    inherit_refs_out:  list = field(default_factory=list)
    instance_attr_types_out: list = field(default_factory=list)
    text_of: Callable = field(default=None, repr=False)

    @property
    def in_class(self) -> bool:
        return self.current_class_id is not None

    def emit_node(self, label, nid, props):
        self.nodes_out.append({"labels": [label], "id": nid, "properties": props})

    def emit_edge(self, etype, src, tgt, **extra):
        entry = {"type": etype, "source": src, "target": tgt}
        entry.update(extra)
        self.edges_out.append(entry)


class SchemaPlugin(ABC):
    @abstractmethod
    def on_class(self, node, ctx): ...
    @abstractmethod
    def on_function(self, node, ctx): ...
    def on_import(self, target_module_id, ctx): pass
    @property
    @abstractmethod
    def call_edge_type(self): ...
    @property
    @abstractmethod
    def inherit_edge_type(self): ...


class SimpleSchema(SchemaPlugin):
    """Module / Class / Function  +  Defines / Calls / Imports / Inherits"""

    @property
    def call_edge_type(self): return "Calls"
    @property
    def inherit_edge_type(self): return "Inherits"

    def on_class(self, node, ctx):
        name = ctx.text_of(node.child_by_field_name("name"))
        if not name:
            return None
        cid = f"{ctx.parent_id}:{name}"   # parent_id, not module_id → correct nesting
        ctx.emit_node("Class", cid, {
            "name":       name,
            "commit":     ctx.commit,
            "file":       ctx.file_,
            "directory":  ctx.directory,
            "start_line": node.start_point[0] + 1,
            "end_line":   node.end_point[0] + 1,
            "source":     ctx.text_of(node)[:8000],
        })
        ctx.emit_edge("Defines", ctx.parent_id, cid)
        return cid

    def on_function(self, node, ctx):
        name = ctx.text_of(node.child_by_field_name("name"))
        if not name:
            return None
        fid = f"{ctx.parent_id}:{name}"
        ctx.emit_node("Function", fid, {
            "name":       name,
            "commit":     ctx.commit,
            "file":       ctx.file_,
            "directory":  ctx.directory,
            "start_line": node.start_point[0] + 1,
            "end_line":   node.end_point[0] + 1,
            "source":     ctx.text_of(node)[:8000],
        })
        ctx.emit_edge("Defines", ctx.parent_id, fid)
        return fid

    def on_import(self, target_module_id, ctx):
        ctx.emit_edge("Imports", ctx.module_id, target_module_id)


SCHEMA = SimpleSchema()
print("Schema loaded: SimpleSchema  (Module / Class / Function | Defines / Calls / Imports / Inherits)")

Schema loaded: SimpleSchema  (Module / Class / Function | Defines / Calls / Imports / Inherits)


## 4. Language Extractor (Tree-Sitter)
We configure the `tree-sitter-python` extractor. This identifies specific node types like function definitions and imports within the raw AST.

In [50]:
from dataclasses import dataclass as _dc
from tree_sitter import Language, Parser
import tree_sitter_python as tspython

try:
    _PY_LANG = Language(tspython.language())
    _parser  = Parser(_PY_LANG)
except TypeError:
    _PY_LANG = Language(tspython.language(), "python")
    _parser  = Parser()
    _parser.set_language(_PY_LANG)


@_dc(frozen=True)
class LanguageExtractor:
    parser: object
    language: object
    builtins: frozenset
    function_types: tuple
    class_types: tuple
    call_type: str
    import_type: str
    from_import_type: str
    assignment_types: tuple
    import_name_types: tuple
    aliased_import_type: str
    import_as_names_type: str
    wildcard_import_type: str
    import_keyword_type: str
    identifier_type: str
    name_field: str
    params_field: str
    superclasses_field: str
    module_name_field: str
    alias_name_field: str
    alias_alias_field: str
    call_function_field: str
    attribute_type: str
    attribute_object_field: str
    attribute_attr_field: str
    attr_name_subfield: str
    self_keywords: frozenset
    source_extensions: tuple = (".py",)
    index_file_name: str = "__init__"


EXTRACTOR = LanguageExtractor(
    parser=_parser,
    language=_PY_LANG,
    builtins=frozenset({
        "print", "len", "range", "str", "int", "float", "bool", "list", "dict",
        "tuple", "set", "type", "isinstance", "issubclass", "hasattr", "getattr",
        "setattr", "delattr", "super", "object", "enumerate", "zip", "map",
        "filter", "sorted", "reversed", "min", "max", "sum", "abs", "round",
        "open", "repr", "iter", "next", "any", "all", "vars", "dir", "id",
        "hash", "callable", "staticmethod", "classmethod", "property",
    }),
    # In tree-sitter-python 0.25+, `async def` is function_definition (no separate node type)
    function_types=("function_definition",),
    class_types=("class_definition",),
    call_type="call",
    import_type="import_statement",
    from_import_type="import_from_statement",
    assignment_types=("assignment", "annotated_assignment", "expression_statement"),
    import_name_types=("dotted_name", "aliased_import"),
    aliased_import_type="aliased_import",
    import_as_names_type="import_as_names",
    wildcard_import_type="wildcard_import",
    import_keyword_type="import",
    identifier_type="identifier",
    name_field="name",
    params_field="parameters",
    superclasses_field="superclasses",
    module_name_field="module_name",
    alias_name_field="name",
    alias_alias_field="alias",
    call_function_field="function",
    attribute_type="attribute",
    attribute_object_field="object",
    attribute_attr_field="attribute",
    attr_name_subfield="",
    self_keywords=frozenset({"self", "cls"}),
)
print("Extractor ready: Python / tree-sitter")

Extractor ready: Python / tree-sitter


## 5. AST Engine Core
The `ASTEngine` performs the heavy lifting: parsing files, building local scopes, and mapping imports to enable cross-file resolution.

In [51]:
import logging
import os
from dataclasses import replace as dc_replace
from tree_sitter import Query, QueryCursor

logger = logging.getLogger(__name__)


def _caps(query, node):
    """Execute a Query and return {capture_name: [Node]} (tree-sitter 0.24+ API)."""
    result = QueryCursor(query).captures(node)
    if isinstance(result, dict):
        return result
    # Pre-0.24 fallback: list of (Node, str) tuples
    d = {}
    for n, name in result:
        d.setdefault(name, []).append(n)
    return d


class ASTEngine:
    """Parse một file .py → nodes + refs cho cross-file resolve."""

    def __init__(self, file_path, repo_path, base_commit, schema, extractor):
        self.extractor   = extractor
        self.file_path   = file_path
        self.repo_path   = repo_path
        self.base_commit = base_commit
        self.schema      = schema
        try:
            self.rel_path = os.path.relpath(file_path, repo_path).replace("\\", "/")
        except ValueError:
            self.rel_path = file_path.replace("\\", "/")
        self.module_id   = f"{self.base_commit}:{self.rel_path}"
        self._source: bytes = b""
        self.import_map: dict[str, str] = {}
        self.from_import_map: dict[str, str | None] = {}

        ext  = extractor
        lang = extractor.language
        self._q_defs = Query(lang,
            "\n".join(f"({t}) @def" for t in (*ext.class_types, *ext.function_types))
        )
        self._q_imports = Query(lang,
            f"({ext.import_type}) @stmt\n({ext.from_import_type}) @stmt"
        )
        self._q_calls   = Query(lang, f"({ext.call_type}) @call")
        self._q_assigns = Query(lang, "(assignment) @assign")

    # ── Public ─────────────────────────────────────────────────────────────────

    def parse(self):
        try:
            with open(self.file_path, "rb") as f:
                self._source = f.read()
        except Exception:
            return None
        try:
            tree = self.extractor.parser.parse(self._source)
        except Exception:
            return None

        commit, _, file_ = self.module_id.partition(":")
        ctx = ParseContext(
            module_id=self.module_id, parent_id=self.module_id,
            commit=commit, file_=file_,
            directory=os.path.dirname(file_) or ".",
            is_module_scope=True, current_class_id=None,
            text_of=self._text,
        )
        ctx.emit_node("Module", self.module_id, {"name": self.rel_path, "commit": self.base_commit})

        self._extract_definitions(tree, ctx)
        self._extract_imports(tree, ctx)
        self._extract_calls(tree, ctx)
        self._extract_self_assigns(tree, ctx)

        local_scope, class_scopes = self._build_scopes(ctx.nodes_out)
        return {
            "module_id":    self.module_id,
            "nodes":        ctx.nodes_out,
            "definite_edges":      ctx.edges_out,
            "call_refs":           ctx.call_refs_out,
            "inherit_refs":        ctx.inherit_refs_out,
            "instance_attr_types": ctx.instance_attr_types_out,
            "import_map":          self.import_map,
            "from_import_map":     self.from_import_map,
            "local_scope":         local_scope,
            "class_scopes":        class_scopes,
        }

    # ── Scope helpers ──────────────────────────────────────────────────────────

    def _scope_of(self, node):
        """ID of the scope that CONTAINS node (walks up node.parent chain)."""
        ext, parts, cur = self.extractor, [], node.parent
        while cur is not None:
            if cur.type in (*ext.class_types, *ext.function_types):
                n = cur.child_by_field_name(ext.name_field)
                if n:
                    parts.append(self._text(n))
            cur = cur.parent
        return ":".join([self.module_id] + list(reversed(parts))) if parts else self.module_id

    def _class_of(self, node):
        """ID of the innermost class enclosing node, or None."""
        ext, cur = self.extractor, node.parent
        while cur is not None:
            if cur.type in ext.class_types:
                n = cur.child_by_field_name(ext.name_field)
                if n:
                    return f"{self._scope_of(cur)}:{self._text(n)}"
            cur = cur.parent
        return None

    # ── Pass 1: class + function definitions ──────────────────────────────────

    def _extract_definitions(self, tree, ctx):
        ext = self.extractor
        def_nodes = sorted(_caps(self._q_defs, tree.root_node).get("def", []),
                           key=lambda n: n.start_byte)
        for node in def_nodes:
            tmp = dc_replace(ctx, parent_id=self._scope_of(node),
                             current_class_id=self._class_of(node))
            if node.type in ext.class_types:
                cid = self.schema.on_class(node, tmp)
                if cid:
                    supers = node.child_by_field_name(ext.superclasses_field)
                    if supers:
                        for arg in supers.children:
                            base = self._extract_base_name(arg)
                            if base:
                                ctx.inherit_refs_out.append({"class_id": cid, "base_name": base})
            else:
                self.schema.on_function(node, tmp)

    # ── Pass 2: imports ────────────────────────────────────────────────────────

    def _extract_imports(self, tree, ctx):
        ext = self.extractor
        for node in _caps(self._q_imports, tree.root_node).get("stmt", []):
            if node.type == ext.import_type:
                for sub in node.children:
                    if sub.type in ext.import_name_types:
                        raw = self._text(sub)
                        mod_str, _, alias = raw.partition(" as ")
                        mod_str = mod_str.strip()
                        alias   = alias.strip() if alias.strip() else mod_str.split(".")[-1]
                        tid = self._resolve_module(mod_str)
                        if tid:
                            self.schema.on_import(tid, ctx)
                            self.import_map[alias] = tid
                            if "." in mod_str:
                                self.import_map[mod_str] = tid
            elif node.type == ext.from_import_type:
                mod_node = node.child_by_field_name(ext.module_name_field)
                if not mod_node:
                    continue
                mod_str    = self._text(mod_node)
                module_tid = self._resolve_module(mod_str)
                if module_tid:
                    self.schema.on_import(module_tid, ctx)
                for local_name, orig_name in self._get_from_imports(node):
                    sub_tid = self._resolve_module(f"{mod_str}.{orig_name}")
                    if sub_tid:
                        self.import_map[local_name] = sub_tid
                    elif module_tid:
                        self.from_import_map[local_name] = f"{module_tid}:{orig_name}"
                    else:
                        self.from_import_map[local_name] = None

    def _get_from_imports(self, from_stmt):
        ext, results, past_import = self.extractor, [], False
        for child in from_stmt.children:
            if child.type == ext.import_keyword_type:
                past_import = True; continue
            if not past_import:
                continue
            if child.type == ext.wildcard_import_type:
                break
            if child.type == ext.identifier_type:
                n = self._text(child); results.append((n, n))
            elif child.type == ext.import_as_names_type:
                for sub in child.children:
                    if sub.type == ext.identifier_type:
                        n = self._text(sub); results.append((n, n))
                    elif sub.type == ext.aliased_import_type:
                        r = self._resolve_aliased(sub)
                        if r: results.append(r)
            elif child.type == ext.aliased_import_type:
                r = self._resolve_aliased(child)
                if r: results.append(r)
        return results

    def _resolve_aliased(self, node):
        ext   = self.extractor
        orig  = node.child_by_field_name(ext.alias_name_field)
        alias = node.child_by_field_name(ext.alias_alias_field)
        if orig:
            o = self._text(orig)
            return (self._text(alias) if alias else o, o)
        return None

    # ── Pass 3: call references ────────────────────────────────────────────────

    def _extract_calls(self, tree, ctx):
        ext = self.extractor
        for call_node in _caps(self._q_calls, tree.root_node).get("call", []):
            func = call_node.child_by_field_name(ext.call_function_field)
            if func is None:
                continue
            base = {
                "caller_id": self._scope_of(call_node),
                "line":      call_node.start_point[0] + 1,
                "file":      self.rel_path,
            }
            if func.type == ext.identifier_type:
                name = self._text(func)
                if name not in ext.builtins:
                    ctx.call_refs_out.append({**base, "kind": "simple", "name": name})

            elif func.type == ext.attribute_type:
                obj  = func.child_by_field_name(ext.attribute_object_field)
                attr = func.child_by_field_name(ext.attribute_attr_field)
                if not (obj and attr):
                    continue
                attr_name = self._text(attr)
                if attr_name in ext.builtins:
                    continue
                obj_text = self._text(obj)
                if obj_text in ext.self_keywords:
                    ctx.call_refs_out.append({
                        **base, "kind": "self_method", "name": attr_name,
                        "class_id": self._class_of(call_node),
                    })
                elif obj.type == ext.attribute_type:
                    inner_obj  = obj.child_by_field_name(ext.attribute_object_field)
                    inner_attr = obj.child_by_field_name(ext.attribute_attr_field)
                    if inner_obj and inner_attr and self._text(inner_obj) in ext.self_keywords:
                        ctx.call_refs_out.append({
                            **base, "kind": "self_attr_method", "name": attr_name,
                            "attr": self._text(inner_attr),
                            "class_id": self._class_of(call_node),
                        })
                    else:
                        ctx.call_refs_out.append({**base, "kind": "attr", "name": attr_name, "obj": obj_text})
                else:
                    ctx.call_refs_out.append({**base, "kind": "attr", "name": attr_name, "obj": obj_text})

    # ── Pass 4: self.attr = Type() assignments ────────────────────────────────

    def _extract_self_assigns(self, tree, ctx):
        ext = self.extractor
        for node in _caps(self._q_assigns, tree.root_node).get("assign", []):
            left  = node.child_by_field_name("left")
            right = node.child_by_field_name("right")
            if not (left and right and left.type == ext.attribute_type and right.type == ext.call_type):
                continue
            obj_node  = left.child_by_field_name(ext.attribute_object_field)
            attr_node = left.child_by_field_name(ext.attribute_attr_field)
            if not (obj_node and attr_node):
                continue
            if self._text(obj_node) not in ext.self_keywords:
                continue
            func = right.child_by_field_name(ext.call_function_field)
            if not func:
                continue
            if func.type == ext.identifier_type:
                type_name = self._text(func)
            elif func.type == ext.attribute_type:
                a = func.child_by_field_name(ext.attribute_attr_field)
                type_name = self._text(a) if a else None
            else:
                continue
            if not type_name or type_name in ext.builtins:
                continue
            class_id = self._class_of(obj_node)
            if class_id:
                ctx.instance_attr_types_out.append({
                    "class_id": class_id,
                    "attr":     self._text(attr_node),
                    "type_name": type_name,
                })

    # ── Module path resolver ───────────────────────────────────────────────────

    def _resolve_module(self, module):
        if module.startswith("."):
            dots     = len(module) - len(module.lstrip("."))
            rel_part = module.lstrip(".")
            base_dir = os.path.dirname(self.rel_path)
            for _ in range(dots - 1):
                base_dir = os.path.dirname(base_dir)
            parts = (
                os.path.join(base_dir, rel_part.replace(".", os.sep)).replace("\\", "/")
                if rel_part else base_dir
            )
        else:
            parts = module.replace(".", "/")
        init_name = self.extractor.index_file_name
        for src_ext in self.extractor.source_extensions:
            for candidate in (f"{parts}{src_ext}", f"{parts}/{init_name}{src_ext}"):
                if os.path.exists(os.path.join(self.repo_path, candidate)):
                    return f"{self.base_commit}:{os.path.relpath(os.path.join(self.repo_path, candidate), self.repo_path).replace(chr(92), '/')}"
        return None

    # ── Scope builder ──────────────────────────────────────────────────────────

    def _build_scopes(self, nodes):
        local_scope, class_scopes = {}, {}
        for node in nodes:
            label   = node["labels"][0]
            if label == "Module":
                continue
            node_id = node["id"]
            name    = node["properties"].get("name", "")
            parts   = node_id[len(self.module_id):].lstrip(":").split(":")
            if len(parts) == 1:
                local_scope[name] = node_id
            elif len(parts) == 2 and label == "Function":
                class_id = f"{self.module_id}:{parts[0]}"
                class_scopes.setdefault(class_id, {})[name] = node_id
        return local_scope, class_scopes

    # ── Helpers ────────────────────────────────────────────────────────────────

    def _text(self, node):
        return self._source[node.start_byte:node.end_byte].decode("utf-8", errors="replace")

    def _extract_base_name(self, node):
        ext = self.extractor
        if node.type == ext.identifier_type:
            return self._text(node)
        if node.type == ext.attribute_type:
            a = node.child_by_field_name(ext.attribute_attr_field)
            return self._text(a) if a else ""
        if node.type in ("subscript", "type"):
            child = node.child_by_field_name("value") or (node.children[0] if node.child_count else None)
            return self._extract_base_name(child) if child else ""
        return ""

    # ── Cross-file resolution — returns ONLY semantic (Calls, Inherits) edges ──

    @staticmethod
    def resolve_cross_file(all_results, schema):
        all_node_ids = set()
        all_node_ids.update(n["id"] for r in all_results for n in r["nodes"])
        fmap   = {r["module_id"]: r["from_import_map"] for r in all_results}
        lscope = {r["module_id"]: r["local_scope"]     for r in all_results}
        cscope = {r["module_id"]: r["class_scopes"]    for r in all_results}
        imap   = {r["module_id"]: r["import_map"]      for r in all_results}
        iat    = {r["module_id"]: r["instance_attr_types"] for r in all_results}

        semantic_edges = []   # Calls + Inherits only

        # Inherit refs
        for r in all_results:
            mid   = r["module_id"]
            ls, im, fim = lscope.get(mid, {}), imap.get(mid, {}), fmap.get(mid, {})
            for ref in r["inherit_refs"]:
                cid, base = ref["class_id"], ref["base_name"]
                target = (
                    ls.get(base)
                    or im.get(base)
                    or (fim.get(base) if fim.get(base) in all_node_ids else None)
                )
                if target and target in all_node_ids:
                    semantic_edges.append({"type": schema.inherit_edge_type, "source": cid, "target": target})

        # Call refs
        for r in all_results:
            mid  = r["module_id"]
            ls   = lscope.get(mid, {})
            im   = imap.get(mid, {})
            fim  = fmap.get(mid, {})
            cs   = cscope.get(mid, {})
            iat_ = {e["class_id"]: e for e in iat.get(mid, [])}

            for ref in r["call_refs"]:
                target = ASTEngine._resolve_call(
                    ref, all_node_ids, ls, im, fim, cs, iat_, cscope, lscope, imap
                )
                if target:
                    semantic_edges.append({
                        "type": schema.call_edge_type,
                        "source": ref["caller_id"],
                        "target": target,
                        "line": ref.get("line"),
                    })

        return semantic_edges

    @staticmethod
    def _resolve_call(ref, all_ids, ls, im, fim, cs, iat, cscope, lscope, imap):
        kind = ref.get("kind", "simple")
        name = ref.get("name", "")

        if kind == "simple":
            return ASTEngine._find_first(name, all_ids, ls, im, fim)

        if kind == "self_method":
            class_id = ref.get("class_id")
            if class_id:
                target = cs.get(class_id, {}).get(name)
                if target and target in all_ids:
                    return target

        if kind == "self_attr_method":
            class_id = ref.get("class_id")
            attr     = ref.get("attr", "")
            if class_id and attr in iat:
                type_name    = iat[attr]["type_name"]
                target_class = ASTEngine._find_first(type_name, all_ids, ls, im, fim)
                if target_class:
                    t = cscope.get(target_class, {}).get(name)
                    if t and t in all_ids:
                        return t

        if kind == "attr":
            obj = ref.get("obj", "")
            module_tid = im.get(obj) or fim.get(obj)
            if module_tid:
                remote_ls = lscope.get(module_tid, {})
                t = remote_ls.get(name)
                if t and t in all_ids:
                    return t
                candidate = imap.get(module_tid, {}).get(name)
                if candidate and candidate in all_ids:
                    return candidate

        return None

    @staticmethod
    def _find_first(name, all_ids, ls, im, fim):
        for source in (ls, im, fim):
            t = source.get(name)
            if t and t in all_ids:
                return t
        return None


print("ASTEngine ready: query-based (Query + QueryCursor, tree-sitter 0.24+)")

ASTEngine ready: query-based (Query + QueryCursor, tree-sitter 0.24+)


## 6. Stage 1: AST Extraction
We iterate through all discovered Python files, running the `ASTEngine` on each to build a collection of local results.

In [52]:
from tqdm.notebook import tqdm

all_results = []
errors      = []

for rel in tqdm(source_files, desc="Parsing"):
    try:
        engine = ASTEngine(
            file_path   = str(Path(REPO_PATH) / rel),
            repo_path   = REPO_PATH,
            base_commit = BASE_COMMIT,
            schema      = SCHEMA,
            extractor   = EXTRACTOR,
        )
        result = engine.parse()
        if result:
            all_results.append(result)
    except Exception as e:
        errors.append((rel, str(e)))

total_nodes = sum(len(r["nodes"]) for r in all_results)
total_refs  = sum(len(r["call_refs"]) for r in all_results)

print(f"Files parsed : {len(all_results)} / {len(source_files)}")
print(f"Nodes        : {total_nodes}")
print(f"Call refs    : {total_refs}")
if errors:
    print(f"Errors       : {len(errors)}  (vd: {errors[0]})")

Parsing:   0%|          | 0/83 [00:00<?, ?it/s]

Files parsed : 83 / 83
Nodes        : 1703
Call refs    : 3469


## 7. Stage 2: Semantic Resolution
In this step, we resolve cross-file references. We connect 'Calls' and 'Inherits' edges by matching names across different modules and class scopes.

In [53]:
semantic_edges = ASTEngine.resolve_cross_file(all_results, SCHEMA)

definite_edges = [e for r in all_results for e in r["definite_edges"]]

print(f"Definite edges  : {len(definite_edges)}  (Defines, Imports)")
print(f"Semantic edges  : {len(semantic_edges)}  (Calls, Inherits)")
print(f"Total edges     : {len(definite_edges) + len(semantic_edges)}")

Definite edges  : 1796  (Defines, Imports)
Semantic edges  : 205  (Calls, Inherits)
Total edges     : 2001


In [ ]:
import json
import numpy as np
from datetime import datetime, timezone
from pathlib import Path
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from collections import defaultdict, deque
from git import Repo as _Repo

_EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"

# ── In-memory caches ──────────────────────────────────────────────────────────
_MODEL_CACHE = {}   # model_id  → SentenceTransformer  (shared across all graphs)
_INDEX_CACHE = {}   # file_path → load_graph_indices result

def _get_model(model_id=_EMBED_MODEL_ID):
    if model_id not in _MODEL_CACHE:
        print(f"Loading embed model: {model_id}")
        _MODEL_CACHE[model_id] = SentenceTransformer(model_id)
    return _MODEL_CACHE[model_id]


def _sig_doc_fn(node):
    source = node.get("source", "").strip()
    if not source:
        return f"{node.get('label','')} {node.get('name','')}"
    lines = source.split("\n")
    sig = []
    for line in lines:
        sig.append(line)
        if line.rstrip().endswith(":"):
            break
    body = "\n".join(lines[len(sig):]).lstrip()
    doc = ""
    for q in ('"""', "'''"):
        if body.startswith(q):
            end = body.find(q, len(q))
            doc = body[len(q): end if end != -1 else len(q) + 400].strip()
            break
    return "\n".join(sig) + ("\n" + doc if doc else "")


def build_graph(commit, repo_path=REPO_PATH, output_dir="/content", force=False,
                embed_model_id=_EMBED_MODEL_ID):
    """
    Checkout commit, parse → resolve → embed → export JSON.
    Returns output_file path. Cache hit nếu file đã tồn tại và force=False.
    """
    short       = commit[:8]
    output_file = f"{output_dir}/flask_graph_{short}.json"

    if not force and Path(output_file).exists():
        print(f"[{short}] disk cache hit → {output_file}")
        return output_file

    # 1. Checkout
    _Repo(repo_path).git.checkout(commit)
    print(f"[{short}] checked out")

    # 2. Source files
    src_files = get_python_files(repo_path)

    # 3. AST parse
    results = []
    for rel in src_files:
        try:
            eng = ASTEngine(str(Path(repo_path) / rel), repo_path, short, SCHEMA, EXTRACTOR)
            r = eng.parse()
            if r:
                results.append(r)
        except Exception:
            pass

    # 4. Resolve cross-file edges
    sem_edges = ASTEngine.resolve_cross_file(results, SCHEMA)
    def_edges = [e for r in results for e in r["definite_edges"]]

    # 5. Flatten nodes / edges
    all_nodes = list({
        node["id"]: {
            "id":         node["id"],
            "label":      node["labels"][0],
            "name":       node["properties"].get("name", ""),
            "file":       node["properties"].get("file", ""),
            "start_line": node["properties"].get("start_line"),
            "end_line":   node["properties"].get("end_line"),
            "source":     node["properties"].get("source", ""),
            "community":  node["properties"].get("community"),
        }
        for r in results for node in r["nodes"]
    }.values())
    all_edges = [
        {"type": e["type"], "source": e["source"], "target": e["target"], "line": e.get("line")}
        for e in (def_edges + sem_edges)
    ]

    # 6. Embeddings — dùng cached model
    _model    = _get_model(embed_model_id)
    vec_cands = [n for n in all_nodes if n["label"] in ("Function", "Class")]
    vecs      = _model.encode(
        [_sig_doc_fn(n) for n in vec_cands], batch_size=64,
        show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True,
    )
    id2emb = {vec_cands[i]["id"]: vecs[i].tolist() for i in range(len(vec_cands))}
    for node in all_nodes:
        node["embedding"] = id2emb.get(node["id"])

    # 7. Export
    graph = {
        "meta": {
            "repo": REPO_URL, "commit": short, "schema": "simple", "lang": "python",
            "built_at": datetime.now(timezone.utc).isoformat(),
            "node_count": len(all_nodes), "edge_count": len(all_edges),
            "embed_model": embed_model_id, "embed_dim": int(vecs.shape[1]),
        },
        "nodes": all_nodes, "edges": all_edges,
    }
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(graph, f, ensure_ascii=False)

    _INDEX_CACHE.pop(output_file, None)

    size_mb = Path(output_file).stat().st_size / 1024 / 1024
    print(f"[{short}] → {output_file}  ({len(all_nodes)} nodes, {len(all_edges)} edges, {size_mb:.1f} MB)")
    return output_file


def load_graph_indices(output_file):
    """
    Load graph JSON, build BM25 + vector + call-graph indices.
    Kết quả được cache in-memory theo output_file — gọi lại không tốn thêm chi phí.
    Trả về dict: nodes, node_by_id, search(), hop_search(), callers(), callees().
    """
    if output_file in _INDEX_CACHE:
        print(f"[{Path(output_file).stem}] memory cache hit")
        return _INDEX_CACHE[output_file]

    print(f"[{Path(output_file).stem}] building indices...")
    with open(output_file, "r", encoding="utf-8") as f:
        gdata = json.load(f)

    nodes          = gdata["nodes"]
    embed_model_id = gdata["meta"].get("embed_model", _EMBED_MODEL_ID)
    node_by_id     = {n["id"]: n for n in nodes}

    # BM25
    corpus = []
    for n in nodes:
        text = n.get("source", "")
        if not text or len(text.strip()) < 5:
            text = f"{n.get('label','')} {n.get('name','')} {n.get('file','')}"
        corpus.append(text.lower().split())
    bm25 = BM25Okapi(corpus)

    # Vector — dùng cached model
    vec_nodes  = [n for n in nodes if n.get("embedding") is not None]
    vec_matrix = np.array([n["embedding"] for n in vec_nodes])
    _model     = _get_model(embed_model_id)

    # Call graph
    fwd = defaultdict(list)
    bwd = defaultdict(list)
    for edge in gdata["edges"]:
        if edge["type"] == "Calls":
            fwd[edge["source"]].append(edge["target"])
            bwd[edge["target"]].append(edge["source"])

    def search(query, n=5, method="hybrid"):
        def _topn(scores, pool):
            if len(pool) <= n:
                return np.argsort(scores)[::-1][:n]
            idx = np.argpartition(scores, -n)[-n:]
            return idx[np.argsort(scores[idx])[::-1]]

        bm_sc = bm25.get_scores(query.lower().split())
        q_emb = _model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
        v_sim = (vec_matrix @ q_emb.T).squeeze()
        vi_of = {n["id"]: i for i, n in enumerate(vec_nodes)}

        if method == "bm25":
            return [{"id": nodes[i]["id"], "name": nodes[i]["name"],
                     "label": nodes[i]["label"], "score": round(float(bm_sc[i]), 4)}
                    for i in _topn(bm_sc, nodes) if bm_sc[i] > 0]

        if method == "vector":
            return [{"id": vec_nodes[i]["id"], "name": vec_nodes[i]["name"],
                     "label": vec_nodes[i]["label"], "score": round(float(v_sim[i]), 4)}
                    for i in _topn(v_sim, vec_nodes)]

        k      = min(50, len(nodes))
        cands  = np.argpartition(bm_sc, -k)[-k:]
        bm_max = float(bm_sc[cands].max()) + 1e-9
        ranked = sorted(
            ((nodes[bi],
              bm_sc[bi] / bm_max * 0.3
              + (float(v_sim[vi_of[nodes[bi]["id"]]]) if nodes[bi]["id"] in vi_of else 0.0) * 0.7)
             for bi in cands),
            key=lambda x: -x[1],
        )
        return [{"id": nd["id"], "name": nd["name"], "label": nd["label"],
                 "score": round(sc, 4)} for nd, sc in ranked[:n]]

    def _bfs(seeds, adj, depth):
        dist = {s: 0 for s in seeds}
        q = deque((s, 0) for s in seeds)
        while q:
            cur, d = q.popleft()
            if d >= depth: continue
            for nxt in adj.get(cur, []):
                if nxt not in dist:
                    dist[nxt] = d + 1
                    q.append((nxt, d + 1))
        return dist

    def callers_fn(node_id, depth=2):
        return _bfs([node_id], bwd, depth)

    def callees_fn(node_id, depth=2):
        return _bfs([node_id], fwd, depth)

    def hop_search_fn(query, n_seeds=5, depth=2, method="hybrid", decay=0.5):
        seeds = search(query, n=n_seeds, method=method)
        if not seeds: return []
        best     = {s["id"]: (0, float(s["score"])) for s in seeds}
        frontier = {s["id"]: float(s["score"]) for s in seeds}
        for hop in range(1, depth + 1):
            nxt_f = {}
            for nid, sc in frontier.items():
                nxt_sc = sc * decay
                for nxt in list(dict.fromkeys(fwd.get(nid, []) + bwd.get(nid, []))):
                    if nxt not in best or best[nxt][1] < nxt_sc:
                        best[nxt] = (hop, nxt_sc)
                    if nxt not in nxt_f or nxt_f[nxt] < nxt_sc:
                        nxt_f[nxt] = nxt_sc
            frontier = nxt_f
        out = []
        for nid, (hop, score) in best.items():
            nd = node_by_id.get(nid)
            if not nd or nd["label"] == "Module": continue
            out.append({"id": nid, "name": nd["name"], "label": nd["label"],
                        "file": nd["file"], "hop": hop, "score": round(score, 4)})
        return sorted(out, key=lambda x: -x["score"])

    result = {
        "nodes": nodes, "node_by_id": node_by_id,
        "search": search, "hop_search": hop_search_fn,
        "callers": callers_fn, "callees": callees_fn,
    }
    _INDEX_CACHE[output_file] = result
    return result


print("build_graph() / load_graph_indices() ready")
print(f"Caches: _MODEL_CACHE={list(_MODEL_CACHE.keys())}, _INDEX_CACHE={list(_INDEX_CACHE.keys())}")


## 8. Export Graph Data
Finally, we consolidate all nodes and edges into a standardized JSON format (`flask_graph.json`) for use in search and visualization.

In [54]:
import json
import numpy as np
from datetime import datetime, timezone
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"

all_nodes = list({
    node["id"]: {
        "id":         node["id"],
        "label":      node["labels"][0],
        "name":       node["properties"].get("name", ""),
        "file":       node["properties"].get("file", ""),
        "start_line": node["properties"].get("start_line"),
        "end_line":   node["properties"].get("end_line"),
        "source":     node["properties"].get("source", ""),
        "community":  node["properties"].get("community"),
    }
    for r in all_results
    for node in r["nodes"]
}.values())

all_edges = [
    {"type": e["type"], "source": e["source"], "target": e["target"], "line": e.get("line")}
    for e in (definite_edges + semantic_edges)
]

# ── Compute & attach embeddings ───────────────────────────────────────────────
def _sig_doc(node):
    source = node.get("source", "").strip()
    if not source:
        return f"{node.get('label','')} {node.get('name','')}"
    lines = source.split("\n")
    sig = []
    for line in lines:
        sig.append(line)
        if line.rstrip().endswith(":"):
            break
    body = "\n".join(lines[len(sig):]).lstrip()
    doc  = ""
    for q in ('"""', "'''"):
        if body.startswith(q):
            end = body.find(q, len(q))
            doc = body[len(q) : end if end != -1 else len(q) + 400].strip()
            break
    return "\n".join(sig) + ("\n" + doc if doc else "")

_embed_model     = SentenceTransformer(EMBED_MODEL_ID)
vec_candidates   = [n for n in all_nodes if n["label"] in ("Function", "Class")]
vec_embeddings   = _embed_model.encode(
    [_sig_doc(n) for n in vec_candidates],
    batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
id_to_emb = {vec_candidates[i]["id"]: vec_embeddings[i].tolist() for i in range(len(vec_candidates))}
for node in all_nodes:
    node["embedding"] = id_to_emb.get(node["id"])

# ── Export ────────────────────────────────────────────────────────────────────
graph = {
    "meta": {
        "repo":        REPO_URL,
        "commit":      BASE_COMMIT,
        "schema":      "simple",
        "lang":        "python",
        "built_at":    datetime.now(timezone.utc).isoformat(),
        "node_count":  len(all_nodes),
        "edge_count":  len(all_edges),
        "embed_model": EMBED_MODEL_ID,
        "embed_dim":   int(vec_embeddings.shape[1]),
    },
    "nodes": all_nodes,
    "edges": all_edges,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(graph, f, ensure_ascii=False)

size_mb = Path(OUTPUT_FILE).stat().st_size / 1024 / 1024
print(f"Exported : {OUTPUT_FILE}")
print(f"  Nodes  : {len(all_nodes)}  (embeddings: {len(vec_candidates)})")
print(f"  Edges  : {len(all_edges)}")
print(f"  Size   : {size_mb:.2f} MB")

# ── Download to local (bỏ comment nếu chạy trên Colab) ──────────────────────
from google.colab import files
files.download(OUTPUT_FILE)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Exported : flask_graph.json
  Nodes  : 1661  (embeddings: 1578)
  Edges  : 2001
  Size   : 14.10 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
import json
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    graph_data = json.load(f)

nodes          = graph_data["nodes"]
EMBED_MODEL_ID = graph_data["meta"].get("embed_model", "BAAI/bge-small-en-v1.5")

# ── BM25 index ────────────────────────────────────────────────────────────────
bm25_corpus, bm25_nodes = [], []
for node in nodes:
    text = node.get("source", "")
    if not text or len(text.strip()) < 5:
        text = f"{node.get('label','')} {node.get('name','')} {node.get('file','')}"
    bm25_corpus.append(text.lower().split())
    bm25_nodes.append(node)

bm25 = BM25Okapi(bm25_corpus)
print(f"BM25 index      : {len(bm25_nodes)} nodes")

# ── Vector index (loaded from JSON) ──────────────────────────────────────────
vec_nodes      = [n for n in nodes if n.get("embedding") is not None]
vec_embeddings = np.array([n["embedding"] for n in vec_nodes])
_embed_model   = SentenceTransformer(EMBED_MODEL_ID)
print(f"Vector index    : {vec_embeddings.shape}  [loaded from JSON]")

# ── Unified search ────────────────────────────────────────────────────────────
def search_graph(query, n=5, method="bm25"):
    """method: 'bm25' | 'vector' | 'hybrid'"""

    def _topn(scores, pool):
        if len(pool) <= n:
            return np.argsort(scores)[::-1][:n]
        idx = np.argpartition(scores, -n)[-n:]
        return idx[np.argsort(scores[idx])[::-1]]

    if method == "bm25":
        sc = bm25.get_scores(query.lower().split())
        return [{"id": bm25_nodes[i]["id"], "name": bm25_nodes[i]["name"],
                 "label": bm25_nodes[i]["label"], "score": round(float(sc[i]), 4),
                 "snippet": bm25_nodes[i].get("source","")[:200]+"..."}
                for i in _topn(sc, bm25_nodes) if sc[i] > 0]

    q_emb  = _embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    v_sims = (vec_embeddings @ q_emb.T).squeeze()

    if method == "vector":
        return [{"id": vec_nodes[i]["id"], "name": vec_nodes[i]["name"],
                 "label": vec_nodes[i]["label"], "score": round(float(v_sims[i]), 4),
                 "snippet": vec_nodes[i].get("source","")[:200]+"..."}
                for i in _topn(v_sims, vec_nodes)]

    if method == "hybrid":
        bm_sc  = bm25.get_scores(query.lower().split())
        k      = min(50, len(bm25_nodes))
        cands  = np.argpartition(bm_sc, -k)[-k:]
        vi_of  = {n["id"]: i for i, n in enumerate(vec_nodes)}
        bm_max = float(bm_sc[cands].max()) + 1e-9
        ranked = sorted(
            ((bm25_nodes[bi],
              bm_sc[bi] / bm_max * 0.3
              + (float(v_sims[vi_of[bm25_nodes[bi]["id"]]]) if bm25_nodes[bi]["id"] in vi_of else 0.0) * 0.7)
             for bi in cands),
            key=lambda x: -x[1],
        )
        return [{"id": nd["id"], "name": nd["name"], "label": nd["label"],
                 "score": round(sc, 4), "snippet": nd.get("source","")[:200]+"..."}
                for nd, sc in ranked[:n]]

    raise ValueError(f"Unknown method: {method!r}")

BM25 index      : 1661 nodes


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector index    : (1578, 384)  [loaded from JSON]


In [ ]:

from collections import defaultdict, deque

# ── Build Calls adjacency lists ───────────────────────────────────────────────
forward  = defaultdict(list)   # caller_id → [callee_id, ...]
backward = defaultdict(list)   # callee_id → [caller_id, ...]

for edge in graph_data["edges"]:
    if edge["type"] == "Calls":
        forward[edge["source"]].append(edge["target"])
        backward[edge["target"]].append(edge["source"])

node_by_id = {n["id"]: n for n in nodes}

print(f"Call graph : {len(forward)} callers  ×  {len(backward)} callees")

# ── BFS helpers ───────────────────────────────────────────────────────────────
def _bfs(seeds, adj, depth):
    dist = {s: 0 for s in seeds}
    q = deque((s, 0) for s in seeds)
    while q:
        cur, d = q.popleft()
        if d >= depth:
            continue
        for nxt in adj.get(cur, []):
            if nxt not in dist:
                dist[nxt] = d + 1
                q.append((nxt, d + 1))
    return dist

def callers(node_id, depth=2):
    """Who calls node_id (BFS backward along Calls edges)."""
    return _bfs([node_id], backward, depth)

def callees(node_id, depth=2):
    """What node_id calls (BFS forward along Calls edges)."""
    return _bfs([node_id], forward, depth)

# ── hop_search: search → seed → BFS expand → decay-score ─────────────────────
def hop_search(query, n_seeds=5, depth=2, method="hybrid", decay=0.5):
    """
    Graph-hop retrieval pipeline:
      1. Find n_seeds via search_graph (BM25 / vector / hybrid).
      2. Layer-by-layer BFS from seeds along both Calls directions.
      3. Score(node) = seed_score × decay^hop_dist.
    Returns list sorted by score desc.
    """
    seeds = search_graph(query, n=n_seeds, method=method)
    if not seeds:
        return []

    # best[nid] = (min_hop, best_score_from_any_seed)
    best = {}
    for s in seeds:
        best[s["id"]] = (0, float(s["score"]))

    frontier = {s["id"]: float(s["score"]) for s in seeds}

    for hop in range(1, depth + 1):
        next_frontier = {}
        for nid, sc in frontier.items():
            nxt_sc = sc * decay
            neighbours = list(dict.fromkeys(forward.get(nid, []) + backward.get(nid, [])))
            for nxt in neighbours:
                if nxt not in best or best[nxt][1] < nxt_sc:
                    best[nxt] = (hop, nxt_sc)
                if nxt not in next_frontier or next_frontier[nxt] < nxt_sc:
                    next_frontier[nxt] = nxt_sc
        frontier = next_frontier

    results = []
    for nid, (hop, score) in best.items():
        node = node_by_id.get(nid)
        if not node or node["label"] == "Module":
            continue
        results.append({
            "id":    nid,
            "name":  node["name"],
            "label": node["label"],
            "file":  node["file"],
            "hop":   hop,
            "score": round(score, 4),
        })

    return sorted(results, key=lambda x: -x["score"])

print("hop_search ready  (callers / callees / hop_search)")


In [ ]:

# ── SWE-bench Flask Demo ──────────────────────────────────────────────────────
# gt_pairs: (function_name, file_substring) — tất cả hàm bị sửa hoặc thêm mới trong patch
SWE_INSTANCES = [
    {
        "id":     "pallets__flask-4074",
        "commit": "a541c2ac8b05c2b23e11bd8540088fce1abc2373",
        "title":  "url_for can't distinguish a blueprint mounted two times",
        "text": (
            "url_for cannot distinguish a blueprint mounted two times. "
            "Blueprint nesting and url_for with relative endpoint names. "
            "register_blueprint url_prefix nested blueprints url_for resolution."
        ),
        "gt_pairs": [
            # app.py — modified
            ("register_blueprint",      "app.py"),
            ("inject_url_defaults",     "app.py"),
            ("preprocess_request",      "app.py"),
            ("process_response",        "app.py"),
            ("update_template_context", "app.py"),
            # blueprints.py — modified
            ("__init__",                "blueprints.py"),
            ("add_url_rule",            "blueprints.py"),
            ("register_blueprint",      "blueprints.py"),
            ("register",                "blueprints.py"),
            # helpers.py — new
            ("_split_blueprint_path",   "helpers.py"),
            # wrappers.py — new
            ("blueprints",              "wrappers.py"),
        ],
    },
    {
        "id":     "pallets__flask-4575",
        "commit": "bd56d19b167822a9a23e2e9e2a07ccccc36baa8d",
        "title":  "Move redirect to the Flask app object",
        "text": (
            "Add a redirect method to the Flask app object. "
            "flask.redirect should look for current_app and call its redirect method "
            "to allow applications to override redirect behavior."
        ),
        "gt_pairs": [
            # app.py
            ("async_to_sync", "app.py"),   # modified
            ("redirect",      "app.py"),   # new
            # helpers.py
            ("external_url_handler", "helpers.py"),  # modified
            ("redirect",             "helpers.py"),  # new
        ],
    },
    {
        "id":     "pallets__flask-4642",
        "commit": "97298e06fe19298c3ff9d2e0ed9ba70bb3fda2c8",
        "title":  "FlaskGroup does not work when nested in a click.group",
        "text": (
            "FlaskGroup does not work when nested in a click group. "
            "FlaskGroup make_context list_commands nested click group context passing."
        ),
        "gt_pairs": [
            # app.py — modified
            ("run",                              "app.py"),
            # cli.py — modified + new
            ("__init__",                         "cli.py"),
            ("list_commands",                    "cli.py"),
            ("make_context",                     "cli.py"),   # new
            ("show_server_banner",               "cli.py"),
            ("routes_command",                   "cli.py"),
            # debughelpers.py — modified
            ("explain_template_loading_attempts","debughelpers.py"),
        ],
    },
]

W = 26
N = 15   # tăng lên 15 vì GT nhiều hơn
summary_rows = []

for inst in SWE_INSTANCES:
    print(f"\n{'='*80}")
    print(f"  {inst['id']}")
    print(f"  {inst['title']}")
    print(f"{'='*80}")

    output_file = build_graph(inst["commit"])
    idx         = load_graph_indices(output_file)

    gt_ids = {
        n["id"]
        for n in idx["nodes"]
        for (fname, ffile) in inst["gt_pairs"]
        if n["name"] == fname and ffile in n.get("file", "")
    }
    n_new = sum(1 for (fname, ffile) in inst["gt_pairs"]
                if not any(n["name"] == fname and ffile in n.get("file","")
                           for n in idx["nodes"]))
    print(f"\n  GT: {len(gt_ids)} nodes found in graph  "
          f"({len(inst['gt_pairs']) - len(gt_ids) + len(gt_ids)} pairs total, "
          f"{n_new} new functions not in graph)")
    for nid in gt_ids:
        nd = idx["node_by_id"].get(nid, {})
        print(f"    {nd.get('name','?'):35s}  {nd.get('file','?')}")

    res_bm25 = idx["search"](inst["text"], n=N, method="bm25")
    res_vec  = idx["search"](inst["text"], n=N, method="vector")
    res_hyb  = idx["search"](inst["text"], n=N, method="hybrid")
    res_hop  = idx["hop_search"](inst["text"], n_seeds=5, depth=2, method="hybrid")[:N]

    def _cell(results, i):
        if i >= len(results): return " " * (W + 2)
        r = results[i]
        mark = "*" if r["id"] in gt_ids else " "
        name = r["name"][:W - 8]
        return f"[{mark}] {name:<{W - 7}} {r.get('score', 0):.3f}"

    print(f"\n  [*] = ground truth")
    print(f"\n  {'#':2}  {'BM25':{W+2}}  {'Vector':{W+2}}  {'Hybrid':{W+2}}  Hop (depth=2)")
    print(f"  {'─'*2}  {'─'*(W+2)}  {'─'*(W+2)}  {'─'*(W+2)}  {'─'*(W+2)}")
    for i in range(N):
        b = _cell(res_bm25, i)
        v = _cell(res_vec,  i)
        h = _cell(res_hyb,  i)
        hop_r = res_hop[i] if i < len(res_hop) else None
        if hop_r:
            mark = "*" if hop_r["id"] in gt_ids else " "
            nm = hop_r["name"][:W - 10]
            hp = f"[{mark}] {nm:<{W-10}} h={hop_r['hop']} {hop_r['score']:.3f}"
        else:
            hp = ""
        print(f"  {i+1:2}  {b}  {v}  {h}  {hp}")

    def recall_at_k(results, k):
        return sum(1 for r in results[:k] if r["id"] in gt_ids) / len(gt_ids) if gt_ids else 0.0

    print(f"\n  Recall@{N}: BM25={recall_at_k(res_bm25,N):.2f}  "
          f"Vector={recall_at_k(res_vec,N):.2f}  "
          f"Hybrid={recall_at_k(res_hyb,N):.2f}  "
          f"Hop={recall_at_k(res_hop,N):.2f}")

    summary_rows.append({
        "id":     inst["id"].split("__")[1],
        "n_gt":   len(gt_ids),
        "recall": [recall_at_k(r, N) for r in (res_bm25, res_vec, res_hyb, res_hop)],
    })

# ── Summary matrix ────────────────────────────────────────────────────────────
print(f"\n\n{'='*80}")
print(f"  SUMMARY MATRIX  —  Recall@{N}  (multi-file GT)")
print(f"{'='*80}")
print(f"  {'Instance':<22}  {'GT':>4}  {'BM25':>6}  {'Vector':>6}  {'Hybrid':>6}  {'Hop':>6}")
print(f"  {'─'*22}  {'─'*4}  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*6}")
for row in summary_rows:
    vals = "  ".join(f"{v:>6.2f}" for v in row["recall"])
    print(f"  {row['id']:<22}  {row['n_gt']:>4}  {vals}")


In [56]:
# ── Demo so sánh 3 phương pháp ────────────────────────────────────────────────
C = 36
for q in [
    "flask application initialization",
    "request context push pop",
    "session cookie signing secret key",
    "error handler 404 not found",
]:
    bm = search_graph(q, method="bm25")
    vc = search_graph(q, method="vector")
    hy = search_graph(q, method="hybrid")
    print(f"\nQuery: '{q}'")
    print(f"  {'BM25':<{C}}  {'Vector (bge-small)':<{C}}  Hybrid")
    print(f"  {'─'*C}  {'─'*C}  {'─'*C}")
    for i in range(5):
        b = f"{bm[i]['name']} ({bm[i]['score']})" if i < len(bm) else ""
        v = f"{vc[i]['name']} ({vc[i]['score']})" if i < len(vc) else ""
        h = f"{hy[i]['name']} ({hy[i]['score']})" if i < len(hy) else ""
        print(f"  {b:<{C}}  {v:<{C}}  {h}")


Query: 'flask application initialization'
  BM25                                  Vector (bge-small)                    Hybrid
  ────────────────────────────────────  ────────────────────────────────────  ────────────────────────────────────
  init_app (9.3203)                     __init__ (0.8639)                     init_app (0.8935)
  __call__ (7.4597)                     init_app (0.8479)                     __call__ (0.7366)
  find_best_app (6.0715)                create_app (0.8322)                   find_best_app (0.6676)
  NoAppException (5.4449)               __init__ (0.8259)                     __init__ (0.644)
  TestNoImports (4.424)                 __init__ (0.8043)                     App (0.6419)

Query: 'request context push pop'
  BM25                                  Vector (bge-small)                    Hybrid
  ────────────────────────────────────  ────────────────────────────────────  ────────────────────────────────────
  from_environ (10.4184)                pus

In [57]:
from collections import Counter

# Node type breakdown
node_counts = Counter(n['label'] for n in all_nodes)
total_nodes = len(all_nodes)

print("=== Node breakdown ===")
for ntype, count in sorted(node_counts.items(), key=lambda x: -x[1]):
    print(f"  {ntype:<12} {count:>5}  ({count/total_nodes*100:.1f}%)")

# Lines of Code (LOC) Stats
func_locs = [n['end_line'] - n['start_line'] + 1 for n in all_nodes if n['label'] == 'Function' and n['start_line'] and n['end_line']]
class_locs = [n['end_line'] - n['start_line'] + 1 for n in all_nodes if n['label'] == 'Class' and n['start_line'] and n['end_line']]

if func_locs:
    print(f"\n  Avg Function LOC : {sum(func_locs)/len(func_locs):.1f}")
if class_locs:
    print(f"  Avg Class LOC    : {sum(class_locs)/len(class_locs):.1f}")

# Edge type breakdown
edge_counts = Counter(e["type"] for e in (definite_edges + semantic_edges))
total_edges = sum(edge_counts.values())
print("\n=== Edge breakdown ===")
for etype, count in sorted(edge_counts.items(), key=lambda x: -x[1]):
    print(f"  {etype:<12} {count:>5}  ({count/total_edges*100:.1f}%)")

print(f"\n  TOTAL EDGES  {total_edges:>5}")

# Import resolution rate
import_edges = edge_counts.get("Imports", 0)
print(f"\n=== Import resolution ===")
print(f"  import_map entries (resolved)  : {sum(len(r.get('import_map',{})) for r in all_results)}")
print(f"  from_import resolved           : {sum(sum(1 for v in r.get('from_import_map',{}).values() if v) for r in all_results)}")
print(f"  from_import UNRESOLVED (None)  : {sum(sum(1 for v in r.get('from_import_map',{}).values() if v is None) for r in all_results)}")
print(f"  Import edges emitted           : {import_edges}")

# Which modules are import targets?
import_targets = Counter(
    e["target"].split(":")[1] if ":" in e["target"] else e["target"]
    for e in definite_edges if e["type"] == "Imports"
)
print(f"\nTop import targets (by file):")
for path, cnt in import_targets.most_common(10):
    print(f"    {cnt:>3}x  {path}")

=== Node breakdown ===
  Function      1429  (86.0%)
  Class          149  (9.0%)
  Module          83  (5.0%)

  Avg Function LOC : 10.8
  Avg Class LOC    : 49.3

=== Edge breakdown ===
  Defines       1620  (81.0%)
  Calls          184  (9.2%)
  Imports        176  (8.8%)
  Inherits        21  (1.0%)

  TOTAL EDGES   2001

=== Import resolution ===
  import_map entries (resolved)  : 0
  from_import resolved           : 48
  from_import UNRESOLVED (None)  : 13
  Import edges emitted           : 176

Top import targets (by file):
     24x  src/flask/helpers.py
     24x  src/flask/globals.py
     20x  src/flask/signals.py
     12x  src/flask/wrappers.py
     11x  src/flask/__init__.py
     11x  src/flask/sansio/scaffold.py
      9x  src/flask/ctx.py
      8x  src/flask/templating.py
      7x  src/flask/testing.py
      7x  src/flask/sansio/app.py
